# Setup & smoke test — Causal NS-AIRun the cells top to bottom. Stop at the first one that fails and report the error.**Before you start:** drag `csai.zip` into the `paper/` folder in the VS Codeexplorer, then run cell 1.

## 1. Unpack the project

In [ ]:
import os, zipfile, pathlibROOT = pathlib.Path.home() / "abtin" / "paper"os.chdir(ROOT)zp = ROOT / "csai.zip"if zp.exists():    with zipfile.ZipFile(zp) as z:        z.extractall(ROOT)    print("unpacked")else:    print("csai.zip not found -- drag it into paper/ first")PROJ = ROOT / "csai"os.chdir(PROJ)print("cwd:", os.getcwd())print(sorted(p.name for p in PROJ.iterdir()))

## 2. Environment checkConfirms the GPU, free VRAM and free disk. BioMistral-7B needs ~15 GB of disk for the download and ~15 GB of VRAM for bf16 weights.

In [ ]:
import shutil, subprocess, sysimport torchprint("python :", sys.version.split()[0])print("torch  :", torch.__version__, "| cuda", torch.version.cuda)print("cuda ok:", torch.cuda.is_available())if torch.cuda.is_available():    free, total = torch.cuda.mem_get_info()    print(f"gpu    : {torch.cuda.get_device_name(0)}")    print(f"vram   : {free/2**30:.1f} GiB free / {total/2**30:.1f} GiB total")    if free / 2**30 < 18:        print("  WARNING: less than 18 GiB free. Close other jobs before the real run.")du = shutil.disk_usage(".")print(f"disk   : {du.free/2**30:.1f} GiB free")if du.free / 2**30 < 40:    print("  WARNING: model download needs ~15 GiB plus HF cache overhead.")

## 3. Install dependencies`torch` is deliberately excluded — yours already works and reinstalling it canbreak the CUDA build.

In [ ]:
!pip install --no-deps -q transformers accelerate datasets safetensors tokenizers!pip install -q scikit-learn scipy numpy pandasimport transformers, sklearn, scipyprint("transformers", transformers.__version__)print("sklearn", sklearn.__version__)

## 4. Fetch real FDA label textThis writes NO clinical thresholds. It only records what the FDA label says andwhere. You and your supervisor fill in the numbers afterwards.Paste the key when prompted — `getpass` keeps it out of the notebook file.

In [ ]:
import getpass, osos.environ["OPENFDA_API_KEY"] = getpass.getpass("openFDA API key: ")

In [ ]:
!python src/fetch_openfda.py --drugs drugs.txt --out data/curation_worksheet.csv

### Inspect what actually came backThe key question: do the labels state explicit numeric thresholds, or only vague phrases like *severe renal impairment*? Read the output before going further.

In [ ]:
import pandas as pdpd.set_option("display.max_colwidth", 200)df = pd.read_csv("data/curation_worksheet.csv")print(f"{len(df)} sentences from {df['query_drug'].nunique()} drugs")print(df.groupby("source_section").size(), "\n")# sentences that contain a number AND a renal/lab cue -- the promising onescue = df["source_text"].str.contains(    r"eGFR|creatinine|clearance|potassium|INR|QT|mL/min|years of age",    case=False, regex=True, na=False)hits = df[df["has_number"] & cue]print(f"{len(hits)} sentences contain both a number and a lab/threshold cue\n")hits[["query_drug", "source_section", "source_text"]].head(20)

## 5. Build the counterfactual datasetUses the interim hand-written rules in `src/rules.py`. Replace once the worksheet is curated.

In [ ]:
!python src/build_dataset.py --out data

## 6. Smoke test — mock backend, no GPUProves the plumbing works. These numbers are **not** results.

In [ ]:
!python src/run_eval.py --backend mock --seeds 0 1!python src/make_table.py --split test

## 7. Real run — one seed firstDownloads ~15 GB on the first call. Start with `batch_size 4`; watch VRAM in aterminal with `watch -n1 nvidia-smi` and raise it if there is headroom.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0!python src/run_eval.py --backend hf \    --model_id BioMistral/BioMistral-7B \    --seeds 0 --batch_size 4 2>&1 | tail -20

### Did the model actually answer in the expected format?`unparsable_rate` above ~5% means the prompt template needs fixing. Look at raw outputs here.

In [ ]:
import jsonrecs = [json.loads(l) for l in open("results/preds_test_base_seed0.jsonl")]bad = [r for r in recs if r["pred"] is None]print(f"unparsable: {len(bad)}/{len(recs)}")for r in recs[:3]:    print("---", r["id"], "| gold:", r["label"], "| pred:", r["pred"])    print(r["raw"][:300])

## 8. Full run — five seedsOnly after cell 7 looks clean. This is the number you show your supervisor.

In [ ]:
!python src/run_eval.py --backend hf \    --model_id BioMistral/BioMistral-7B \    --seeds 0 1 2 3 4 --batch_size 4 2>&1 | tail -30!python src/make_table.py --split test

### Generalisation check on the two held-out rule families

In [ ]:
!python src/run_eval.py --backend hf \    --model_id BioMistral/BioMistral-7B \    --seeds 0 1 2 --batch_size 4 --split heldout 2>&1 | tail -20!python src/make_table.py --split heldout

## 9. RQ3 control — perplexity baselineVariants 2–4 do not touch the weights, so this is unchanged by construction. Record it now as the pre-intervention baseline for Aim 3.

In [ ]:
!python src/perplexity.py --model_id BioMistral/BioMistral-7B --n 200

---## What to send back if something breaks- the full traceback- `nvidia-smi` output taken *while* the job runs- three sample `raw` strings from cell 7